In [1]:
import os
from dotenv import load_dotenv
from tavily import TavilyClient
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

# 1. Load environment variables from .env
load_dotenv()

# Verify presence of required keys
required_keys = ["GEMINI_API_KEY", "GROQ_API_KEY", "TAVILY_API_KEY"]
for key in required_keys:
    status = "Loaded" if os.getenv(key) else "MISSING"
    print(f"{key}: {status}")

GEMINI_API_KEY: Loaded
GROQ_API_KEY: Loaded
TAVILY_API_KEY: Loaded


# Document Loading & PDF Parsing

Concept: LangChain document loaders standardize raw data sources (PDFs, HTML, Markdown) into Document objects containing page_content (string) and metadata (dict). We use PyPDFLoader to parse local PDFs into clean string context.

In [2]:
file_path = './asdasd.docx'

if file_path.endswith('pdf'):
    print('pdf')
elif file_path.endswith(('.doc', 'docx')):
    print('doccccc')
else:
    print('NONE')


doccccc


In [12]:
import re

def load_resume_text(file_path: str) -> str:
    """Parses a multi-page PDF resume or docx resume into a consolidated text string."""

    if file_path.endswith('.pdf'):
        from langchain_community.document_loaders import PyMuPDFLoader

        loader = PyMuPDFLoader(file_path)
        pages = loader.load()

        raw_text = "\n\n".join([page.page_content for page in pages])

        # Clean up non-breaking spaces and white-space gaps
        resume_context = re.sub(r"\xa0", " ", raw_text)

        return resume_context
    elif file_path.endswith(('.doc', 'docx')):
        from langchain_community.document_loaders import UnstructuredWordDocumentLoader

        loader = UnstructuredWordDocumentLoader(file_path)
        pages = loader.load()

        raw_text = "\n\n".join([page.page_content for page in pages])

        # Clean up non-breaking spaces and redundant whitespace
        resume_context = re.sub(r"\xa0", " ", raw_text)
        resume_context = re.sub(r"\n\s*\n", "\n\n", resume_context)

        return resume_context
    
    else:
        pass




In [13]:
# Example usage:
resume_context = load_resume_text("./resume.pdf")

print(f"Parsed Resume Length: {len(resume_context)} characters")
print("-"*100)
print(resume_context)

Parsed Resume Length: 6334 characters
----------------------------------------------------------------------------------------------------
Hamza Ahmed Siddiqui 
Data Scientist & ML Engineer · Computer Vision & NLP · AI Solutions 
hamzaahmedsiddiqui10@gmail.com | +49 1575 5794618 | Göttingen, Germany | LinkedIn | GitHub 
 
WoRk ExPERIEncE 
~Research Assistant - Computer Vision                                                                                           Apr 2026 – Present 
Eckerlab                                                                                                                                                Göttingen, Germany 
Project: Video Generalization Dataset 
•​ Investigated cross-domain generalization in video action recognition on Something-Something V2 (SSv2) by 
benchmarking feature representations from self-supervised vision models (DINOv2, V-JEPA Base/Giant) with 
trained attention heads.    
•​ Integrated Segment Anything Model (SAM 3) into the vi

# Tool Integration with Tavily Search API
Concept: An agent's tool acts as an external context retriever. To avoid scraping blocks or enterprise OAuth setup, we query search engines with restricted domain filters (site:[linkedin.com/jobs](https://linkedin.com/jobs), site:[indeed.com/viewjob](https://indeed.com/viewjob)) to pull live job descriptions directly.

In [ ]:
def fetch_target_jobs(role_title: str, location: str = "Germany", max_results: int = 3) -> str:
    """Fetches and formats real-time job descriptions from Tavily API."""
    client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
    
    # Target exact job portals to retrieve actual JD snippets
    search_query = f"site:linkedin.com/jobs OR site:indeed.com '{role_title}' {location}"
    
    response = client.search(
        query=search_query, 
        max_results=max_results, 
        search_depth="advanced"
    )
    
    job_snippets = []
    for i, result in enumerate(response.get("results", []), 1):
        snippet = (
            f"--- JOB POSTING {i}: {result.get('title')} ---\n"
            f"URL: {result.get('url')}\n"
            f"Description: {result.get('content')}\n"
        )
        job_snippets.append(snippet)
        
    return "\n".join(job_snippets)

# Example test run:
# job_data = fetch_target_jobs("Machine Learning Engineer", "Germany")
# print(job_data[:400])